In [ ]:
# -*- coding: utf-8 -*-
r"""
NPC 3D Spatial Dose–Anatomy Project
Radiotherapy and Oncology

Purpose
-------
A. Development cohort — Calibration (M3)
B. Independent external validation cohort — Calibration (M3)
C. Development cohort — DCA (M1, M2-Abl, M3-Abl, M3)
D. Independent external validation cohort — DCA (M1, M2-Abl, M3-Abl, M3)

This notebook renders Figure 3 from the calibration and decision-curve workbooks generated by evaluation/calibration_and_dca_analysis.py. It preserves the observed quantile-bin calibration points and Wilson intervals without cosmetic smoothing, uses panel labels A–D, and exports PDF, TIFF, PNG, source data, and an audit record.

Run this entire notebook cell directly in Jupyter.
"""

from pathlib import Path
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# 0. PATHS / SETTINGS
# ============================================================

def require_env_path(name: str) -> Path:
    """Return a required absolute path from an environment variable."""
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Set {name} to an absolute path before running this notebook."
        )
    path = Path(value).expanduser()
    if not path.is_absolute():
        raise RuntimeError(f"{name} must be an absolute path: {value!r}")
    return path.resolve()

ROOT = require_env_path("NPC_PROJECT_ROOT")

CAL_DIR = ROOT / "calibration_FINAL_v3"
DCA_DIR = ROOT / "DCA_FINAL_v1"

CAL_XLSX = CAL_DIR / "NPC_3DCNN_Calibration_FINAL_v3.xlsx"
DCA_XLSX = DCA_DIR / "NPC_3DCNN_DCA_FINAL_v1.xlsx"

OUT_DIR = ROOT / "Figure3_RnO_FINAL_v4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_CAL_SHEETS = {
    "Calibration_metrics",
    "Calibration_bins",
    "M3_predictions_used",
}
EXPECTED_DCA_SHEETS = {
    "DCA_all_thresholds",
    "Reference_thresholds",
    "Input_prediction_QA",
}

MAIN_DCA_MODELS = ["M1", "M2-Abl", "M3-Abl", "M3"]
DISPLAY_THRESH_MIN = 0.05
DISPLAY_THRESH_MAX = 0.80
DCA_YMIN = -0.05
DCA_YMAX = 0.50

# DCA model styling is locked to Figure 2 for manuscript-wide consistency.
# Figure 2 scheme:
#   M1      = blue solid
#   M2-Abl  = orange dashed
#   M3-Abl  = red solid
#   M3      = brown solid, emphasized
# Reference strategies remain black/gray.
DCA_STYLE = {
    "Treat none": {"color": "#000000", "linestyle": "-",  "linewidth": 1.0},
    "Treat all":  {"color": "#7F7F7F", "linestyle": "--", "linewidth": 1.0},
    "M1":         {"color": "#1F77B4", "linestyle": "-",  "linewidth": 1.5},
    "M2-Abl":     {"color": "#FF7F0E", "linestyle": "--", "linewidth": 1.5},
    "M3-Abl":     {"color": "#D62728", "linestyle": "-",  "linewidth": 1.7},
    "M3":         {"color": "#8C564B", "linestyle": "-",  "linewidth": 2.8},
}

# Calibration panel style
CAL_IDEAL_COLOR = "#7F7F7F"
# M3 calibration uses the same manuscript-wide M3 color as Figure 2 and DCA.
# The connecting line, markers, and error bars share the same hue.
CAL_OBS_COLOR = "#8C564B"

# Approximate R&O two-column width
FIG_W = 7.48
FIG_H = 7.35

# ============================================================
# 1. HELPERS
# ============================================================

def logit_stable(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def expit_stable(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required file not found:\n{path}")
    return path

def read_excel_sheets(path):
    path = require_file(path)
    xls = pd.ExcelFile(path)
    return xls, set(xls.sheet_names)

# ============================================================
# 2. LOAD FROZEN RESULTS
# ============================================================

cal_xls, cal_sheets = read_excel_sheets(CAL_XLSX)
missing = EXPECTED_CAL_SHEETS - cal_sheets
if missing:
    raise KeyError(
        "Calibration workbook is missing sheets: "
        + ", ".join(sorted(missing))
    )

dca_xls, dca_sheets = read_excel_sheets(DCA_XLSX)
missing = EXPECTED_DCA_SHEETS - dca_sheets
if missing:
    raise KeyError(
        "DCA workbook is missing sheets: "
        + ", ".join(sorted(missing))
    )

cal_metrics = pd.read_excel(CAL_XLSX, sheet_name="Calibration_metrics")
cal_bins = pd.read_excel(CAL_XLSX, sheet_name="Calibration_bins")
m3_pred = pd.read_excel(CAL_XLSX, sheet_name="M3_predictions_used")

dca_all = pd.read_excel(DCA_XLSX, sheet_name="DCA_all_thresholds")
dca_ref = pd.read_excel(DCA_XLSX, sheet_name="Reference_thresholds")
dca_input_qa = pd.read_excel(DCA_XLSX, sheet_name="Input_prediction_QA")

# ============================================================
# 3. BASIC QA
# ============================================================

# Calibration
for cohort, expected_n in [("Development", 309), ("External", 180)]:
    sub = m3_pred[m3_pred["cohort"] == cohort].copy()
    if len(sub) != expected_n:
        raise ValueError(
            f"M3_predictions_used has {len(sub)} rows for {cohort}; "
            f"expected {expected_n}."
        )

for cohort in ["Development", "External"]:
    m = cal_metrics[
        (cal_metrics["cohort"] == cohort)
        & (cal_metrics["model"] == "M3")
    ]
    if len(m) != 1:
        raise ValueError(
            f"Calibration_metrics must contain exactly one M3 row for {cohort}."
        )

    b = cal_bins[
        (cal_bins["cohort"] == cohort)
        & (cal_bins["model"] == "M3")
    ]
    if len(b) < 5:
        raise ValueError(
            f"Calibration_bins for M3/{cohort} appears too sparse."
        )

# DCA
for cohort, expected_n in [("Development", 309), ("External", 180)]:
    for model in MAIN_DCA_MODELS:
        sub = dca_input_qa[
            (dca_input_qa["cohort"] == cohort)
            & (dca_input_qa["model"] == model)
        ]
        if len(sub) != 1:
            raise ValueError(
                f"Input_prediction_QA must contain exactly one row for "
                f"{cohort}/{model}."
            )
        n = int(sub.iloc[0]["n"])
        if n != expected_n:
            raise ValueError(
                f"Input_prediction_QA gives n={n} for {cohort}/{model}; "
                f"expected {expected_n}."
            )

# ============================================================
# 4. BUILD SMOOTH CALIBRATION CURVES (M3 ONLY)
# ============================================================

smooth_rows = []

for cohort in ["Development", "External"]:
    m = cal_metrics[
        (cal_metrics["cohort"] == cohort)
        & (cal_metrics["model"] == "M3")
    ].iloc[0]

    p_grid = np.linspace(0.001, 0.999, 300)
    lp = logit_stable(p_grid)

    # Use the already frozen joint logistic calibration intercept/slope
    # to create a smooth calibration curve for display.
    cal_intercept = float(m["calibration_intercept"])
    cal_slope = float(m["calibration_slope"])
    y_hat = expit_stable(cal_intercept + cal_slope * lp)

    tmp = pd.DataFrame({
        "cohort": cohort,
        "predicted_probability": p_grid,
        "smoothed_observed_probability": y_hat,
    })
    smooth_rows.append(tmp)

smooth_df = pd.concat(smooth_rows, ignore_index=True)

# ============================================================
# 5. PREPARE DCA DISPLAY DATA
# ============================================================

dca_plot = dca_all[
    dca_all["threshold_probability"].between(
        DISPLAY_THRESH_MIN,
        DISPLAY_THRESH_MAX,
        inclusive="both"
    )
].copy()

for cohort in ["Development", "External"]:
    for model in MAIN_DCA_MODELS:
        sub = dca_plot[
            (dca_plot["cohort"] == cohort)
            & (dca_plot["model"] == model)
        ]
        if len(sub) == 0:
            raise ValueError(
                f"No DCA plotting rows found for {cohort}/{model} in "
                f"{DISPLAY_THRESH_MIN:.2f}–{DISPLAY_THRESH_MAX:.2f}."
            )

# ============================================================
# ============================================================

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8.5,
    "axes.titlesize": 9.5,
    "axes.labelsize": 8.5,
    "legend.fontsize": 7.8,
    "xtick.labelsize": 7.8,
    "ytick.labelsize": 7.8,
})

fig, axes = plt.subplots(2, 2, figsize=(FIG_W, FIG_H))
axA, axB = axes[0, 0], axes[0, 1]
axC, axD = axes[1, 0], axes[1, 1]

# ---------- Calibration panels ----------
def plot_calibration_panel(ax, cohort, panel_label):
    bins_sub = cal_bins[
        (cal_bins["cohort"] == cohort)
        & (cal_bins["model"] == "M3")
    ].copy()

    met = cal_metrics[
        (cal_metrics["cohort"] == cohort)
        & (cal_metrics["model"] == "M3")
    ].iloc[0]

    # Ideal calibration line
    ax.plot(
        [0, 1], [0, 1],
        linestyle="--",
        color=CAL_IDEAL_COLOR,
        linewidth=1.0,
        label="Ideal"
    )

    # Observed quantile-bin calibration.
    # Keep the original frozen bin summaries; do NOT smooth or refit them.
    # A thin connecting line is used only as a visual guide.
    yerr = np.vstack([
        bins_sub["observed_fraction"] - bins_sub["observed_ci_low"],
        bins_sub["observed_ci_high"] - bins_sub["observed_fraction"],
    ])

    ax.plot(
        bins_sub["mean_predicted"],
        bins_sub["observed_fraction"],
        color=CAL_OBS_COLOR,
        linewidth=0.95,
        marker="o",
        markersize=4.2,
        label="Observed"
    )

    ax.errorbar(
        bins_sub["mean_predicted"],
        bins_sub["observed_fraction"],
        yerr=yerr,
        fmt="none",
        ecolor=CAL_OBS_COLOR,
        elinewidth=0.7,
        capsize=2.0,
        alpha=0.70
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Observed event probability")
    ax.set_title(
        "Development cohort" if cohort == "Development"
        else "Independent external validation cohort"
    )
    ax.grid(alpha=0.18, linewidth=0.5)

    # Use unambiguous wording: Brier score + Calibration-in-the-large + slope
    txt = (
        f"Brier score = {met['brier_score']:.3f}\n"
        f"Calibration-in-the-large = {met['calibration_in_the_large']:.2f}\n"
        f"Calibration slope = {met['calibration_slope']:.2f}"
    )
    ax.text(
        0.04, 0.96, txt,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=7.3
    )

    ax.text(
        -0.13, 1.05, panel_label,
        transform=ax.transAxes,
        fontsize=10.5,
        fontweight="bold",
        va="top"
    )

    ax.legend(
        loc="lower right",
        frameon=False
    )

plot_calibration_panel(axA, "Development", "A")
plot_calibration_panel(axB, "External", "B")

# ---------- DCA panels ----------
def plot_dca_panel(ax, cohort, panel_label):
    sub = dca_plot[dca_plot["cohort"] == cohort].copy()

    # so take them from the first prespecified model.
    base = sub[sub["model"] == MAIN_DCA_MODELS[0]].copy()

    ax.plot(
        base["threshold_probability"],
        base["net_benefit_treat_none"],
        color=DCA_STYLE["Treat none"]["color"],
        linestyle=DCA_STYLE["Treat none"]["linestyle"],
        linewidth=DCA_STYLE["Treat none"]["linewidth"],
        label="Treat none"
    )

    ax.plot(
        base["threshold_probability"],
        base["net_benefit_treat_all"],
        color=DCA_STYLE["Treat all"]["color"],
        linestyle=DCA_STYLE["Treat all"]["linestyle"],
        linewidth=DCA_STYLE["Treat all"]["linewidth"],
        label="Treat all"
    )

    for model in MAIN_DCA_MODELS:
        s = sub[sub["model"] == model].copy()
        ax.plot(
            s["threshold_probability"],
            s["net_benefit_model"],
            color=DCA_STYLE[model]["color"],
            linestyle=DCA_STYLE[model]["linestyle"],
            linewidth=DCA_STYLE[model]["linewidth"],
            label=model
        )

    ax.set_xlim(DISPLAY_THRESH_MIN, DISPLAY_THRESH_MAX)
    ax.set_ylim(DCA_YMIN, DCA_YMAX)
    ax.set_xlabel("Threshold probability")
    ax.set_ylabel("Net benefit")
    ax.set_title(
        "Development cohort" if cohort == "Development"
        else "Independent external validation cohort"
    )
    ax.grid(alpha=0.18, linewidth=0.5)

    ax.text(
        -0.13, 1.05, panel_label,
        transform=ax.transAxes,
        fontsize=10.5,
        fontweight="bold",
        va="top"
    )

plot_dca_panel(axC, "Development", "C")
plot_dca_panel(axD, "External", "D")

# Shared legend for DCA only
handles_dca, labels_dca = axD.get_legend_handles_labels()
fig.legend(
    handles_dca,
    labels_dca,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=6,
    frameon=False
)

# Spacing for R&O-style double-column figure
fig.subplots_adjust(
    left=0.09,
    right=0.98,
    top=0.97,
    bottom=0.12,
    wspace=0.22,
    hspace=0.30
)

# Save
for ext, dpi in [("png", 600), ("tiff", 600), ("pdf", 300)]:
    fig.savefig(
        OUT_DIR / f"Figure3_RnO_FINAL_v4.{ext}",
        dpi=dpi,
        bbox_inches="tight"
    )

plt.show()
plt.close(fig)

# ============================================================
# 7. EXPORT SOURCE DATA / AUDIT
# ============================================================

# Prepare figure-specific source data
cal_bins_export = cal_bins[
    cal_bins["model"] == "M3"
].copy()

cal_metrics_export = cal_metrics[
    cal_metrics["model"] == "M3"
].copy()

dca_plot_export = dca_plot[
    dca_plot["model"].isin(MAIN_DCA_MODELS)
].copy()

source_xlsx = OUT_DIR / "Figure3_RnO_FINAL_v4_source_data.xlsx"

with pd.ExcelWriter(source_xlsx, engine="openpyxl") as writer:
    cal_metrics_export.to_excel(
        writer,
        sheet_name="Calibration_metrics_M3",
        index=False
    )
    cal_bins_export.to_excel(
        writer,
        sheet_name="Calibration_bins_M3",
        index=False
    )
    dca_plot_export.to_excel(
        writer,
        sheet_name="DCA_plot_data_main4",
        index=False
    )
    dca_ref[
        dca_ref["model"].isin(MAIN_DCA_MODELS)
    ].to_excel(
        writer,
        sheet_name="DCA_reference_thresholds",
        index=False
    )

audit = {
    "version": "Figure3_RnO_FINAL_v4",
    "uses_existing_results_only": True,
    "calibration_input_dir": str(CAL_DIR),
    "dca_input_dir": str(DCA_DIR),
    "outputs": {
        "figure_png": str(OUT_DIR / "Figure3_RnO_FINAL_v4.png"),
        "figure_tiff": str(OUT_DIR / "Figure3_RnO_FINAL_v4.tiff"),
        "figure_pdf": str(OUT_DIR / "Figure3_RnO_FINAL_v4.pdf"),
        "source_data": str(source_xlsx),
    },
    "figure_structure": {
        "A": "Development cohort — Calibration (M3)",
        "B": "Independent external validation cohort — Calibration (M3)",
        "C": "Development cohort — DCA (M1, M2-Abl, M3-Abl, M3)",
        "D": "Independent external validation cohort — DCA (M1, M2-Abl, M3-Abl, M3)",
    },
    "dca_display_range": {
        "x_min": DISPLAY_THRESH_MIN,
        "x_max": DISPLAY_THRESH_MAX,
        "y_min": DCA_YMIN,
        "y_max": DCA_YMAX,
        "note": "Display range only; DCA results themselves remain unchanged."
    },
    "calibration_display_note": (
        "Frozen observed quantile-bin points with 95% CI are retained and "
        "connected by a thin guide line. Error bars use the same series color "
        "as the observed calibration points. No smoothing or recalibration is "
        "used for the displayed observed calibration curve."
    ),
    "dca_display_note": (
        "Main DCA panels use the same model color and line-style scheme as Figure 2 "
        "so that M1, M2-Abl, M3-Abl, and M3 retain identical visual identities "
        "across the main-text performance figures."
    ),
    "status": "PASS"
}

with open(
    OUT_DIR / "Figure3_RnO_FINAL_v4_audit.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(audit, f, ensure_ascii=False, indent=2)

# ============================================================
# ============================================================

print("\n" + "=" * 92)
print(f"Output folder:\n{OUT_DIR}")
print("\nGenerated files:")
print("  1) Figure3_RnO_FINAL_v4.pdf")
print("  2) Figure3_RnO_FINAL_v4.tiff")
print("  3) Figure3_RnO_FINAL_v4.png")
print("  4) Figure3_RnO_FINAL_v4_source_data.xlsx")
print("  5) Figure3_RnO_FINAL_v4_audit.json")
print("\nFigure structure:")
print("  A  Development cohort — Calibration (M3)")
print("  B  Independent external validation cohort — Calibration (M3)")
print("  C  Development cohort — DCA (M1, M2-Abl, M3-Abl, M3)")
print("  D  Independent external validation cohort — DCA (M1, M2-Abl, M3-Abl, M3)")
print("=" * 92)